In [ ]:
"""
This script is designed to run in Google Colab, not Ocean Code, to show you how we download the data
because it depends on Google Colab-specific features and environment settings.
"""
import ee
import time
import logging
import os
import math
'''
EE_PROJECT=YOUR_EE_PROJECT_ID
EXPORT_BASE_DIR=./exports
'''

BASE_DIR = os.getenv("EXPORT_BASE_DIR", "./exports")
LOG_DIR = os.path.join(BASE_DIR, "logs")
OUTPUT_DIR = os.path.join(BASE_DIR, "Mangrove")

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

logging.basicConfig(
    filename=os.path.join(LOG_DIR, "export_log.txt"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

EE_PROJECT = os.getenv("EE_PROJECT", "YOUR_EE_PROJECT_ID")
ee.Initialize(project=EE_PROJECT)

dataset = ee.Image("JCU/Murray/GIC/global_tidal_wetland_change/2019")


def tile_generator():
    """Yield rectangular export tiles in EPSG:4326."""
    lon_step = 18
    lat_step = 13.268

    lon_min, lon_max = -180, 180
    lat_min, lat_max = -66.34, 66.34

    n_lon = int((lon_max - lon_min) / lon_step)
    n_lat = int(math.ceil((lat_max - lat_min) / lat_step))

    for i in range(n_lon):
        west = lon_min + i * lon_step
        east = west + lon_step

        for j in range(n_lat):
            south = lat_min + j * lat_step
            north = min(lat_min + (j + 1) * lat_step, lat_max)

            yield (
                i,
                j,
                ee.Geometry.Rectangle(
                    [west, south, east, north],
                    proj="EPSG:4326",
                    geodesic=False
                )
            )


class TaskManager:
    """Track and poll Earth Engine export tasks."""

    def __init__(self, max_concurrent=3):
        self.max_concurrent = max_concurrent
        self.tasks = []

    def submit(self, task, task_id):
        task.start()
        self.tasks.append(task)
        logging.info(f"STARTED {task_id}")

    def poll(self):
        alive = []
        for task in self.tasks:
            status = task.status()
            state = status.get("state", "UNKNOWN")

            if state == "COMPLETED":
                logging.info(f"COMPLETED {task.id}")
            elif state == "FAILED":
                logging.error(f"FAILED {task.id}: {status.get('error_message')}")
            else:
                alive.append(task)

        self.tasks = alive


def export_band(band_name, max_concurrent=3):
    """Export a dataset band tile by tile to Google Drive."""
    print(f"=== Exporting band: {band_name} ===")
    mgr = TaskManager(max_concurrent)

    for i, j, tile in tile_generator():
        tile_id = f"{band_name}_x{i}_y{j}"
        marker_file = os.path.join(OUTPUT_DIR, f"{tile_id}.tif")

        if os.path.exists(marker_file):
            continue

        export_params = {
            "image": dataset.select(band_name).clip(tile),
            "description": tile_id,
            "folder": "EarthEngineExports",
            "fileNamePrefix": tile_id,
            "region": tile,
            "scale": 100,
            "crs": "EPSG:4326",
            "maxPixels": 1e13,
            "fileFormat": "GeoTIFF",
            "skipEmptyTiles": True,
        }

        while True:
            mgr.poll()
            if len(mgr.tasks) < mgr.max_concurrent:
                task = ee.batch.Export.image.toDrive(**export_params)
                mgr.submit(task, tile_id)
                print(f"Submitted {tile_id}")
                break
            time.sleep(5)

    while mgr.tasks:
        mgr.poll()
        print(f"Running tasks: {len(mgr.tasks)}")
        time.sleep(10)

    print(f"=== Finished band: {band_name} ===")


if __name__ == "__main__":
    export_band("lossYear", max_concurrent=3)
    # export_band("gainYear", max_concurrent=3)
    print("All exports finished.")

In [ ]:
'''check tif'''

import os
import rasterio
import numpy as np
import pandas as pd

BASE_DIR = os.getenv("EXPORT_BASE_DIR", "./exports")
LOG_DIR = os.path.join(BASE_DIR, "logs")
OUTPUT_DIR = os.path.join(BASE_DIR, "Mangrove")
BAND_NAME = os.getenv("BAND_NAME", "lossYear")

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

AUDIT_CSV = os.path.join(BASE_DIR, f"{BAND_NAME}_tif_audit.csv")

records = []

for fn in sorted(os.listdir(OUTPUT_DIR)):
    if not fn.lower().endswith(".tif"):
        continue

    path = os.path.join(OUTPUT_DIR, fn)

    try:
        with rasterio.open(path) as ds:
            arr = ds.read(1)
            nodata = ds.nodata

            valid_mask = ~np.isnan(arr)
            if nodata is not None:
                valid_mask &= (arr != nodata)

            valid_pixels = int(np.count_nonzero(valid_mask))

            if valid_pixels > 0:
                valid_values = arr[valid_mask]
                min_val = float(valid_values.min())
                max_val = float(valid_values.max())
            else:
                min_val = None
                max_val = None

            records.append({
                "file": fn,
                "shape": arr.shape,
                "valid_pixels": valid_pixels,
                "min": min_val,
                "max": max_val,
                "crs": str(ds.crs),
                "dtype": str(arr.dtype),
                "nodata": nodata
            })

    except Exception as e:
        records.append({
            "file": fn,
            "shape": None,
            "valid_pixels": -1,
            "min": None,
            "max": None,
            "crs": None,
            "dtype": None,
            "nodata": None,
            "error": str(e)
        })

df = pd.DataFrame(records)
df.to_csv(AUDIT_CSV, index=False)

print(f"Audit completed: {AUDIT_CSV}")